In [11]:
from transformers import BertTokenizer , BertModel

import numpy as np 
import torch 
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
model_name = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21658.83it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
documents = [
    "A small bakery in the city center has introduced a new collection of handmade chocolates and traditional desserts. Customers can also attend weekend cooking workshops.",

    "Recent developments in battery technology have made electric vehicles more practical. Modern batteries can store more energy, allowing cars to travel longer distances without recharging.",

    "The city council has announced plans to construct several new parks in residential neighborhoods. These green spaces will include walking paths, playgrounds, and recreational areas.",

    "A group of scientists discovered a rare species of butterfly in a tropical forest. The insect has unusual wing patterns and can survive in extremely humid environments.",

    "Public transportation systems in large cities often include buses, trains, and underground metro networks. Many governments are investing in modern transportation infrastructure to improve urban mobility.",

    "The purple elephant danced beside a floating refrigerator while seven invisible bananas discussed the temperature of the moon.",

    "Environmental researchers have discovered that planting trees in urban environments can help reduce air pollution and lower temperatures. Trees absorb carbon dioxide and provide shade.",

    "Many employees have started working from home instead of traveling to traditional offices. This allows workers to spend more time with their families and reduce daily transportation expenses.",

    "Heavy rainfall caused significant disruptions across the city yesterday. Several major roads became flooded, forcing drivers to take alternative routes. Public buses experienced long delays because of traffic congestion, leaving passengers stranded at bus stops.",

    "A local university has introduced a new online education platform. Students can attend lectures, complete assignments, and communicate with instructors through a digital learning environment.",

    "Weather forecasting technologies have improved significantly in recent years. Scientists now use satellites, radar systems, and machine learning models to predict storms and changes in atmospheric conditions.",

    "The wooden spaceship ordered a cup of invisible coffee while a talking mountain tried to solve a mathematical equation about dancing clouds.",

    "A technology company has developed a new smartphone with an advanced camera system. The device uses artificial intelligence to improve image quality and battery efficiency.",

    "Transportation researchers are studying how electric buses can reduce air pollution in urban areas. Several cities have introduced environmentally friendly vehicles into their public transit networks.",

    "Extreme weather events such as hurricanes and snowstorms can cause significant damage to buildings and agricultural fields. Farmers often experience financial losses when crops are destroyed.",

    "A famous musician announced an international concert tour featuring performances in several European countries. Tickets are expected to become available next month."
]


# QUERY
query = "How did extreme weather conditions affect public transportation and daily travel in the city?"



In [ ]:
def get_embedding(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    last_hidden_state = outputs.last_hidden_state

  
    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    embedding = (
        (last_hidden_state * attention_mask).sum(dim=1)
        / attention_mask.sum(dim=1).clamp(min=1)
    )

    return embedding.cpu().numpy()

In [24]:
doc_embedding = np.vstack([get_embedding(doc) for doc in documents])

query_embedding = get_embedding(query)

similarities = cosine_similarity(query_embedding, doc_embedding)[0]

# Büyükten küçüğe sırala
ranked_indices = np.argsort(similarities)[::-1]

for rank, i in enumerate(ranked_indices, start=1):
    print(f"Rank {rank} | Document {i} : {similarities[i]:.4f}")

Rank 1 | Document 13 : 0.7533
Rank 2 | Document 4 : 0.7245
Rank 3 | Document 10 : 0.7176
Rank 4 | Document 14 : 0.7154
Rank 5 | Document 6 : 0.7137
Rank 6 | Document 8 : 0.6964
Rank 7 | Document 9 : 0.6812
Rank 8 | Document 7 : 0.6729
Rank 9 | Document 11 : 0.6719
Rank 10 | Document 3 : 0.6697
Rank 11 | Document 5 : 0.6503
Rank 12 | Document 1 : 0.6450
Rank 13 | Document 2 : 0.6446
Rank 14 | Document 0 : 0.6278
Rank 15 | Document 15 : 0.6263
Rank 16 | Document 12 : 0.6145
